Gets the raw market data from Gamma API

In [ ]:
import requests
import pandas as pd
import time
import gc

BASE_URL = "https://gamma-api.polymarket.com/markets/keyset"

CHUNK_SIZE = 500_000

all_markets = []

cursor = None
page = 1
chunk = 1
total_downloaded = 0

while True:

    params = {
        "limit": 100,
        "closed": "true"
    }

    if cursor is not None:
        params["after_cursor"] = cursor

    print(f"Downloading page {page}...")

    r = requests.get(BASE_URL, params=params, timeout=30)
    r.raise_for_status()

    data = r.json()
    markets = data["markets"]

    if not markets:
        break

    all_markets.extend(markets)
    total_downloaded += len(markets)

    print(f"Received {len(markets)}")
    print(f"Total downloaded: {total_downloaded}")
    print()

    # Save chunk
    if len(all_markets) >= CHUNK_SIZE:

        print(f"\nSaving chunk {chunk}...")

        pd.DataFrame(all_markets).to_parquet(
            f"polymarket_closed_markets_part_{chunk}.parquet",
            index=False
        )

        # Completely clear memory
        del all_markets
        gc.collect()
        all_markets = []

        chunk += 1

        print("Memory cleared.\n")

    cursor = data.get("next_cursor")

    if cursor is None:
        print("Reached end.")
        break

    page += 1
    time.sleep(0.05)

# Save any remaining markets
if all_markets:

    pd.DataFrame(all_markets).to_parquet(
        f"polymarket_closed_markets_part_{chunk}.parquet",
        index=False
    )

print(f"Downloaded {total_downloaded} markets.")